# Project 18 — Dynamics & Drift (Local-Level State-Space Model)

**Scenario.** A sensor / growth-curve time series whose *true level drifts* over time. We observe the level through measurement noise. We want the latent trajectory **and** an honest split of the two noise sources.

**New skill.** Temporal dependence and latent states. **Key pitfall.** The **process** noise (how much the level wanders) and the **observation** noise (measurement error) are *confounded* — both make the observed series wiggly. Separating them needs informative priors and enough data.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240601

## Step 1 — Problem & data-generating story

Local-level model: the latent level follows a Gaussian random walk, and each observation is the level plus noise.

$$\text{level}_t = \text{level}_{t-1} + w_t,\; w_t\sim N(0,\sigma_\text{lvl}),\qquad y_t = \text{level}_t + e_t,\; e_t\sim N(0,\sigma_\text{obs}).$$

**Assumptions:** (a) the level evolves as a random walk (no mean reversion); (b) both noises are Gaussian and constant; (c) observations are conditionally independent given the level. We know the truths $\sigma_\text{lvl}=0.30$, $\sigma_\text{obs}=0.60$.

In [ ]:
from data.generate_data import generate
data = generate()
y = data['y']
print(f"T={data['t']}, true sigma_level={data['truth']['sigma_level']}, "
      f"true sigma_obs={data['truth']['sigma_obs']}")
fig, ax = plt.subplots(figsize=(7,3.5))
ax.plot(data['level_true'], 'k--', lw=1.5, label='true latent level')
ax.plot(y, '.', color='#4C72B0', alpha=0.7, label='observed y')
ax.set(xlabel='time', ylabel='value', title='Latent level vs noisy observations')
ax.legend(); plt.tight_layout()

## Step 2 — Model specification (non-centred random walk + justified priors)

We build the random walk **non-centred**: standardised innovations $z_t\sim N(0,1)$ scaled by $\sigma_\text{lvl}$ and cumulatively summed. This avoids the funnel that a centred random walk develops as $\sigma_\text{lvl}\to 0$.

**Priors.** $\sigma_\text{lvl}\sim\text{HalfNormal}(0.5)$ (expect modest drift), $\sigma_\text{obs}\sim\text{HalfNormal}(1.0)$ (measurement error can be larger). These are deliberately *not* both vague — vague priors let the two variances trade off freely (see the broken notebook).

In [ ]:
from model import build_model, fit, build_ar1_model, fit_ar1
model = build_model(data)
model

## Step 3 — Prior predictive checks

We simulate series implied by the priors. We want plausible drifting series of the right rough magnitude — not explosive random walks (process prior too wide) nor flat lines (too tight).

In [ ]:
rng = np.random.default_rng(RNG)
fig, ax = plt.subplots(figsize=(7,3.5))
for _ in range(6):
    sl = abs(rng.normal(0,0.5)); so = abs(rng.normal(0,1.0))
    lvl = 5 + np.cumsum(rng.normal(0, sl, size=data['t']))
    ax.plot(lvl + rng.normal(0, so, size=data['t']), lw=1)
ax.set(xlabel='time', ylabel='y', title='Prior predictive series — plausible drift')
plt.tight_layout()

## Step 4 — Inference (NUTS)

Settings: `draws=600, tune=1000, chains=2, target_accept=0.95, cores=1`. The non-centred parameterisation plus a raised `target_accept` keep the random-walk geometry divergence-free. (`cores=1` because multiprocess sampling can hang without a linked BLAS.)

In [ ]:
idata = fit(data, draws=600, tune=1000, chains=2, seed=101)

## Step 5 — Computational diagnostics

Check $\hat R\approx 1.00$, ESS, and divergences (want 0). Look at the **joint** posterior of $(\sigma_\text{lvl}, \sigma_\text{obs})$: a strong negative correlation is the signature of the variance confounding — with our priors and T=100 it should be present but mild, and both should bracket their truths.

In [ ]:
print(az.summary(idata, var_names=['sigma_level','sigma_obs','level0']))
print('divergences:', int(idata.sample_stats['diverging'].sum()))

In [ ]:
az.plot_trace(idata, var_names=['sigma_level','sigma_obs']); plt.tight_layout()

In [ ]:
az.plot_pair(idata, var_names=['sigma_level','sigma_obs'], kind='scatter',
             scatter_kwargs={'alpha':0.2}); plt.tight_layout()

## Step 6 — Posterior predictive checks & latent trajectory

Overlay the inferred latent level (posterior mean + band) on the truth and the data. A good fit tracks the true level inside a band that is tighter than the observation scatter (the model 'sees through' the noise).

In [ ]:
lvl = idata.posterior['level']
lvl_mean = lvl.mean(dim=('chain','draw')).values
lvl_lo = lvl.quantile(0.03, dim=('chain','draw')).values
lvl_hi = lvl.quantile(0.97, dim=('chain','draw')).values
fig, ax = plt.subplots(figsize=(7,3.8))
ax.fill_between(np.arange(data['t']), lvl_lo, lvl_hi, color='#4C72B0', alpha=0.25,
                label='94% level band')
ax.plot(lvl_mean, color='#4C72B0', label='inferred level')
ax.plot(data['level_true'], 'k--', label='true level')
ax.plot(data['y'], '.', color='gray', alpha=0.5, label='data')
ax.set(xlabel='time', ylabel='value', title='Latent level recovery')
ax.legend(); plt.tight_layout()

In [ ]:
mae = float(np.mean(np.abs(lvl_mean - data['level_true'])))
print(f'latent-level recovery MAE = {mae:.3f}')

## Step 7 — Model criticism & comparison (AR(1) alternative)

Is a persistent random-walk drift even needed? We compare against a stationary **AR(1)** model via LOO. The local-level model should be competitive or better when the level genuinely drifts; AR(1) assumes mean reversion. We use `az.compare` on the two `InferenceData` objects (both carry log-likelihood).

In [ ]:
idata_ar1 = fit_ar1(data, draws=600, tune=1000, chains=2, seed=101)
cmp = az.compare({'local_level': idata, 'ar1': idata_ar1}, ic='loo')
print(cmp[['rank','elpd_loo','p_loo','dse']])

## Step 8 — Decision & communication

Translate into a decision: e.g. the current level estimate (last time point) with its credible interval, and the probability the level is trending up over the final stretch — the kind of statement a process engineer acts on.

In [ ]:
last = idata.posterior['level'].isel(level_dim_0=-1).values.ravel()
slope = (idata.posterior['level'].isel(level_dim_0=-1).values
         - idata.posterior['level'].isel(level_dim_0=-11).values).ravel()
print(f'current level = {last.mean():.2f} '
      f'(94% [{np.percentile(last,3):.2f}, {np.percentile(last,97):.2f}])')
print(f'P(level rose over last 10 steps) = {np.mean(slope>0):.2f}')

**Conclusion (for a collaborator).** We separated genuine drift from measurement noise: the latent level is recovered within MAE < 0.5, and both noise scales bracket their true values. Report the *level* and its trend, not the raw noisy readings. See `summary_onepager.md`.